# 07 过拟合与正则化

前面我们讲了完整训练流程，也讲了优化器怎么更新参数。

但训练并不是让训练集 loss 越低越好这么简单。一个模型可能在训练集上表现很好，却在新数据上一塌糊涂。

这一节要讲的就是：**模型为什么会过拟合，以及正则化为什么能缓解过拟合。**

## 1. 先重新理解训练的目标

训练模型的目标表面上是最小化训练损失：

$$
\min_{\theta}\mathcal{L}_{train}
$$

但真正的目标不是让模型只在训练集上好，而是希望它在没见过的新数据上也好。

也就是说，我们真正关心的是泛化能力。

训练集只是模型学习规律的材料，不是最终考试题。

## 2. 什么是过拟合

过拟合指的是：模型在训练集上表现很好，但在验证集或测试集上表现不好。

典型现象是：

$$
\mathcal{L}_{train}\downarrow
$$

但：

$$
\mathcal{L}_{val}\uparrow
$$

训练损失越来越低，说明模型越来越会处理训练数据。

验证损失越来越高，说明模型越来越不会处理新数据。

所以过拟合不是“模型没学会”，而是“模型学得太细，学到了训练集里不该学的偶然细节”。

## 3. 为什么会过拟合

过拟合通常来自三个原因。

第一个原因是模型太复杂。参数太多、层太深、神经元太多时，模型表达能力很强。表达能力强是好事，但也意味着它有能力把训练集里的噪声和偶然细节记下来。

第二个原因是数据太少。如果训练数据很少，模型看到的样本不够丰富，它就容易把少数样本里的特殊现象误认为普遍规律。

第三个原因是训练太久。模型一开始可能先学到主要规律，继续训练太久后，可能开始学习训练集里的细枝末节。

可以把训练过程粗略理解成：

```text
先学大规律
-> 再学小规律
-> 最后可能连噪声也学进去
```

## 4. 什么是欠拟合

欠拟合和过拟合相反。

欠拟合指模型连训练集都学不好。

典型现象是：

$$
\mathcal{L}_{train}\text{ 很高}
$$

$$
\mathcal{L}_{val}\text{ 也很高}
$$

这说明模型能力不够，或者训练方式有问题。

欠拟合像是题目本身都没学会。

过拟合像是只背熟了训练题，但不会做新题。

## 5. 什么是正则化

正则化是一类方法，目标是限制模型不要过度复杂，从而提升泛化能力。

如果只看训练损失，模型会拼命让训练集误差变小：

$$
\min_{\theta}\mathcal{L}_{train}
$$

正则化会在目标里加入一个“别太复杂”的约束：

$$
\min_{\theta}\left(\mathcal{L}_{train}+\lambda\Omega(\theta)\right)
$$

其中：

- $\mathcal{L}_{train}$ 表示训练损失。
- $\Omega(\theta)$ 表示模型复杂度惩罚。
- $\lambda$ 控制惩罚强度。

这句话的意思是：模型不仅要在训练集上错得少，还不能用太复杂、太极端的方式去做到这一点。

## 6. 权重衰减是什么

权重衰减是一种常见正则化方法，也常被称为 $L_2$ 正则化。

它的想法是：不要让权重变得太大。

普通训练目标是：

$$
\mathcal{L}_{train}
$$

加入 $L_2$ 正则化后：

$$
\mathcal{L}_{total}=\mathcal{L}_{train}+\lambda\sum_i w_i^2
$$

这里的：

$$
\sum_i w_i^2
$$

会惩罚过大的权重。

权重越大，这一项越大，模型就要付出更高代价。

## 7. 为什么大权重容易过拟合

权重大，意味着输入稍微变化一点，输出就可能变化很多。

比如一个简单模型：

$$
y=wx+b
$$

如果 $w$ 很大，$x$ 轻微变化就会导致 $y$ 大幅变化。

在高维神经网络中，大权重可能让模型对训练样本里的细节特别敏感。它可能为了拟合某几个特殊样本，把决策边界扭得很复杂。

权重衰减的作用，就是温和地告诉模型：

```text
不要为了训练集里的少数细节，把参数调得太极端
```

所以权重衰减不是直接提高训练集表现，而是牺牲一点训练集拟合能力，换取更好的泛化能力。

## 8. Dropout 是什么

Dropout 是神经网络里非常经典的正则化方法。

它的做法是：训练时随机让一部分神经元暂时不工作。

假设某一层隐藏表示是：

$$
\mathbf{h}=[h_1,h_2,h_3,h_4]
$$

Dropout 可能随机把其中一些位置变成 $0$：

$$
\tilde{\mathbf{h}}=[h_1,0,h_3,0]
$$

这不是因为这些神经元没用，而是故意让网络不要过度依赖某几个神经元。

## 9. Dropout 为什么能缓解过拟合

如果没有 Dropout，模型可能形成很强的依赖：某个神经元专门记住某种训练集细节，后面的层也依赖它。

Dropout 训练时随机关闭神经元，相当于不断告诉模型：

```text
不要指望某个固定神经元永远存在
```

这样模型被迫学习更分散、更稳健的表示。

可以把 Dropout 理解成一种训练扰动。模型在带扰动的情况下仍然要做对，就不容易只记住训练集里的脆弱细节。

注意：Dropout 通常只在训练时使用。验证和测试时，不再随机关闭神经元。

## 10. 早停是什么

早停的英文是 Early Stopping。

它的想法非常直观：如果验证集表现开始变差，就不要继续训练了。

训练过程可能是这样：

```text
前期：训练 loss 下降，验证 loss 也下降
中期：训练 loss 继续下降，验证 loss 基本稳定
后期：训练 loss 继续下降，验证 loss 开始上升
```

后期就可能是模型开始过拟合训练集。

早停就是在验证集表现最好的时候保存模型，而不是盲目训练到最后。

## 11. 数据增强是什么

数据增强是从数据角度缓解过拟合。

如果训练数据太少，模型容易记住具体样本。数据增强的想法是：在不改变标签的前提下，制造更多合理变化。

图像任务里常见增强包括：

- 随机裁剪
- 水平翻转
- 颜色扰动
- 旋转
- 缩放

比如猫的图片左右翻转后，仍然是猫。

数据增强让模型看到更多变化，从而不容易只记住训练图片的固定位置、固定颜色、固定背景。

## 12. 模型复杂度控制

除了直接加正则化，还可以通过控制模型复杂度来缓解过拟合。

比如：

- 减少层数。
- 减少每层神经元数量。
- 减少参数量。
- 使用更简单的模型结构。

模型太简单会欠拟合。

模型太复杂又容易过拟合。

所以模型复杂度不是越高越好，而是要和数据量、任务难度相匹配。

## 13. Batch Normalization 算正则化吗

Batch Normalization 的主要目的不是正则化，而是让训练更稳定、更容易。

它会对一层的中间输出做标准化，缓解训练过程中分布变化带来的问题。

不过，在实际训练中，Batch Normalization 有时也会带来一点正则化效果，因为每个 batch 的统计量带有轻微噪声。

所以可以这样记：

```text
BatchNorm 的主业是稳定训练
顺带可能有一点正则化效果
```

BatchNorm 本身值得单独讲，后面会专门展开。

## 14. 如何判断该用哪种方法

先看训练曲线。

如果训练 loss 和验证 loss 都很高，优先考虑欠拟合：

- 增加模型能力。
- 训练更久。
- 调整学习率。
- 检查数据和标签。

如果训练 loss 很低，但验证 loss 很高，优先考虑过拟合：

- 增加数据或数据增强。
- 使用权重衰减。
- 使用 Dropout。
- 使用早停。
- 降低模型复杂度。

不要一看到效果不好就随便加技巧。先判断是欠拟合还是过拟合，再决定怎么处理。

## 15. 常见正则化方法对比

| 方法 | 核心思想 | 主要作用 |
|---|---|---|
| 权重衰减 | 惩罚过大的权重 | 让模型不要太极端 |
| Dropout | 随机关闭神经元 | 减少对局部特征的依赖 |
| 早停 | 验证集变差就停止 | 防止训练太久学噪声 |
| 数据增强 | 制造合理变化 | 让模型见到更多情况 |
| 降低模型复杂度 | 减少参数和表达能力 | 降低记忆训练集的能力 |

这些方法不是互斥的。实际项目中经常组合使用。

## 16. 本节总结

这一节的逻辑链是：

```text
训练目标不是只让训练集表现好
-> 真正目标是泛化能力
-> 模型太复杂、数据太少、训练太久会导致过拟合
-> 正则化就是限制模型不要过度记忆训练集
-> 权重衰减限制权重大小
-> Dropout 降低神经元依赖
-> 早停避免训练太久
-> 数据增强让模型见到更多变化
```

先记住一句话：正则化不是为了让训练集 loss 更低，而是为了让模型在新数据上更可靠。

下一节可以继续讲 Batch Normalization：为什么深层网络训练会不稳定，BatchNorm 又是怎么让训练更顺的。